## Retrieving Articles

### XML Sources

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
#import undetected_chromedriver as uc
from pyvirtualdisplay import Display
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

# Gathering XML articles
#def retrieve_xml_articles(xml_links: list):

display = Display(visible=0, size=(1920, 1080))
display.start()

#chrome_options = uc.ChromeOptions()
chrome_options = Options()
#chrome_options.add_argument("--headless")                  # Run in headless mode
#chrome_options.add_argument("--disable-gpu")               # Disable GPU (usually helpful on Windows)
chrome_options.add_argument("--no-sandbox")  

driver = webdriver.Chrome(options=chrome_options)

# Initalizing news articles df  
new_articles = pd.DataFrame(columns=["title", "link", "source", "language", "published_date", "keywords"])  

xml_links = ["https://apnews.com/news-sitemap-content.xml"]

for url in xml_links:
    driver.get(url)
    time.sleep(3)  # let the Cloudflare challenge resolve
    xml_site = driver.page_source
    soup = BeautifulSoup(xml_site, "html.parser")

    articles = soup.find_all("url")

    for article in articles:

        title = article.find("news:title").get_text() if article.find("news:title") else None # There's an example of an AP article not having a title huh who knew
        link = article.find("loc").text # get_text doesn't work here for links 
        source = article.find("news:name").get_text()
        language = article.find("news:language").get_text()
        published_date = article.find("news:publication_date").get_text()
        #last_modified = article.find("lastmod").get_text(strip=True) if article.find("lastmod") else None  Getting rid of last modified date because AP was only source using it
        keywords = article.find("news:keywords").get_text(strip=True) if article.find("news:keywords") else None


        tempdf = pd.DataFrame({
        "title": title,
        "link": link,
        "source": source,
        "language": language,
        "published_date": published_date,
        #"last_modified": last_modified,
        "keywords": [keywords]}, index=[0]) 

        new_articles = pd.concat([new_articles, tempdf], ignore_index=True)

driver.quit()
display.stop()

# Use .replace when replacing entire value, else use str.replace which incoroprates regex
new_articles["language"] = new_articles["language"].replace("en", "eng") 

new_articles = new_articles[new_articles['language'] == 'eng']

#    return new_articles

FileNotFoundError: [WinError 2] The system cannot find the file specified

In [5]:
print(soup)

<html xmlns="http://www.w3.org/1999/xhtml"><head><style id="xml-viewer-style">/* Copyright 2014 The Chromium Authors
 * Use of this source code is governed by a BSD-style license that can be
 * found in the LICENSE file.
 */

:root {
  color-scheme: light dark;
}

div.header {
    border-bottom: 2px solid black;
    padding-bottom: 5px;
    margin: 10px;
}

@media (prefers-color-scheme: dark) {
  div.header {
    border-bottom: 2px solid white;
  }
}

div.folder &gt; div.hidden {
    display:none;
}

div.folder &gt; span.hidden {
    display:none;
}

.pretty-print {
    margin-top: 1em;
    margin-left: 20px;
    font-family: monospace;
    font-size: 13px;
}

#webkit-xml-viewer-source-xml {
    display: none;
}

.opened {
    margin-left: 1em;
}

.comment,
.cdata {
    white-space: pre;
}

.folder-button {
    user-select: none;
    cursor: pointer;
    display: inline-block;
    margin-left: -10px;
    width: 10px;
    background-repeat: no-repeat;
    background-position: left top;


In [9]:
print(ap)

<Response [403]>


In [2]:
from sqlalchemy import text, create_engine

engine = create_engine("sqlite:///articles.db")

with engine.connect() as connection:
    xml_links = pd.read_sql_query(
        text("""SELECT article_links 
                FROM article_sources
                where source_type == 'XML'"""), connection)

xml_links_list = xml_links['article_links'].tolist()

new_articles = retrieve_xml_articles(xml_links_list)

### RSS Feeds

In [6]:
from sqlalchemy import text, create_engine

engine = create_engine("sqlite:///articles.db")

with engine.connect() as connection:
    rss_links = pd.read_sql_query(
        text("""SELECT article_links FROM article_sources
                WHERE source_type == 'RSS' AND status == 1""" ), connection)
    
rss_urls_list = rss_links['article_links'].tolist()

rss_urls_list

['https://community.fabric.microsoft.com/oxcrx34285/rss/board?board.id=fbc_pbiupdatesblog',
 'https://feeds.content.dowjones.io/public/rss/RSSWorldNews',
 'https://feeds.content.dowjones.io/public/rss/RSSWSJD',
 'https://feeds.arstechnica.com/arstechnica/index',
 'https://www.nasa.gov/feed/',
 'https://github.com/pandas-dev/pandas/releases.atom',
 'https://github.com/numpy/numpy/releases.atom',
 'https://github.com/psf/requests/releases.atom',
 'https://blog.python.org/rss.xml',
 'https://github.com/astral-sh/uv/releases.atom',
 'http://github.com/ollama/ollama/releases.atom',
 'http://github.com/super-productivity/super-productivity/releases.atom']

In [ ]:
import feedparser
import pandas as pd


rss_stories = pd.DataFrame(columns=["title", "summary", "link", "published_date", "source"])

for rss_url in rss_urls_list:

    # Parse the feed
    feed = feedparser.parse(rss_url)

    for entry in feed.entries: # Figure out way to limit entires

        tempdf = pd.DataFrame({
        "title": entry.title,
        "link": entry.link,
        "summary": entry.get("description"),
        "published_date": entry.get("published") or entry.get("updated"),
        "source": feed.channel.title}, index=[0]) 
        
        rss_stories = pd.concat([rss_stories, tempdf], ignore_index=True)



AttributeError: object has no attribute 'description'

In [12]:
for entry in feed.entries:
    entry.get("description")
    

In [4]:
from sqlalchemy import text, create_engine

engine = create_engine("sqlite:///articles.db")

with engine.connect() as connection:
    rss_links = pd.read_sql_query(
        text("""SELECT article_links
                FROM article_sources
                where source_type == 'RSS'""" ), connection)
    
rss_links_list = rss_links['article_links'].tolist()

rss_stories = retrieve_rss_feeds(rss_links_list)

## Concatenate/Clean RSS and XML Results

In [5]:
import pandas as pd
from dateutil import parser

combined_articles = pd.concat([new_articles, rss_stories], ignore_index=True)

# Normalizing published dates in a really clean way (reconciling date formats like "2026-01-01T04:43:20-05:00", "2025-12-31T16:58:19-05:00", "Thu, 20 Nov 2025 13:36:00 Z", and "Thu, 25 Dec 2025 04:00:00 GMT")
def normalize_date(x):
    return parser.parse(x)

# Normalizing published dates to consistent datetime format (keeps timezone differences)
combined_articles['published_date'] = combined_articles['published_date'].apply(normalize_date)

# Connverting all dates into proper datetimes and UTC for consistency (UTC = True converts dates that are not utc to utc)
combined_articles['published_date'] = pd.to_datetime(combined_articles['published_date'], utc=True)

In [6]:
# Extracting keywords for specific articles using the links variable 
def extract_keywords(row):
    source = row['source']
    link = row['link']

    # Associated Press
    if source == "Associated Press":
        if "article/" in link:
            keywords = ", ".join(link.split("article/")[1].split("-")[:-1])
            return keywords
        else:
            return None

    # WSJ sources
    elif source in ["WSJ.com: World News", "Technology - WSJ.com"]:
        keywords = ", ".join(link.split("/")[-1].split("-")[:-1])
        return keywords

    # All other sources
    else:
        return row['keywords']

# Apply the function
combined_articles['keywords'] = combined_articles.apply(extract_keywords, axis=1)

In [7]:
# Mapping a category to each article based on keywords in title, summary, and keywords columns
import re

#def extract_categories(df: pd.DataFrame):

# Category keywords mapping (keywords need to be lowercase because that's what I'm matching to)
category_keywords = {
    "China": ["china", "beijing", "xi", "taiwan", "alibaba", "tencent", "huawei", "baidu", "hong kong", "shanghai"],
    "Technology": ["ai", "data", "cloud", "software", "technology", "nvidia", "intel", "amd"],
    "Politics": ["election", "government", "policy", "congress", "economy", "protest", "protesters", "military"],
    "International": ["global", "united nations", "diplomacy", "european union", "eu", "nato", "un", "african union", "war"
                    "asean", "brics", "arab league", "opec", "latin america", "middle east", "shanghai cooperation organization"]
    }

combined_articles['category'] = None

for i, row in combined_articles.iterrows():

    # Safely build combined text (handle NaN)
    title = '' if pd.isna(row.get('title')) else str(row['title'])
    summary = '' if pd.isna(row.get('summary')) else str(row['summary'])
    keywords = '' if pd.isna(row.get('keywords')) else str(row['keywords'])
    combined_text = " ".join([title, keywords, summary]).lower() # Make everything lowercase for matching

    # Initialize matched_category state to none
    matched_category = None

    for category, kw_list in category_keywords.items():
        if any(re.search(rf"\b{kw}\b", combined_text) for kw in kw_list):
            matched_category = category
            combined_articles.at[i, 'category'] = matched_category  # Use .at instead of loc or iloc
            break # exiting the loop on first match

    # Default fallback
    if not matched_category:
        combined_articles.at[i, 'category'] = "Other"

#return df

In [8]:
# Manually input categories for specific sources depending on how I want to show them

## manually input a separate category for coding related notices
pattern = r"Python Insider|Release notes|Power BI Updates Blog"
combined_articles.loc[combined_articles["source"].str.contains(pattern, case=False, na=False), "category"] = "Code"

In [9]:
# Filtering out unnessary articles
## Filtering out all news articles that didn't get assigned a caetegory (df name changes to signify clean)
cleaned_articles = combined_articles[combined_articles['category'] != 'Other']

## Apparently the AP categorizes lotery articles as 'data' so I'm manually filtering them out here
cleaned_articles = cleaned_articles[~cleaned_articles["title"].str.contains("Winning numbers|Sports Betting Line|Gulf States Sportswatch Daily Listings", case=False, na=False)]

# Capitalizes all the keywords that are in list format
cleaned_articles.loc[:, 'keywords'] = cleaned_articles['keywords'].apply(
    lambda x: ", ".join(w.strip().capitalize() for w in x.split(","))
    if isinstance(x, str) else x)

In [10]:
from zoneinfo import ZoneInfo
from datetime import datetime

# Using a .copy to avoid SettingWithCopyWarning when I add new variable
cleaned_articles = cleaned_articles.copy()

# Supressing microseconds when I get todays date
today = datetime.now(tz=ZoneInfo("UTC")).strftime('%Y-%m-%d %H:%M:%S') + ".000000"

# Applying todaysd date as the uploaded date 
cleaned_articles["upload_date"] = today

# Keep coding related articles if today os Tuesday, Thursday, or Saturday
approved_days = [1,3,5]

if datetime.strptime(today.split(".")[0], "%Y-%m-%d %H:%M:%S").weekday() in approved_days:
    pass
else:
    cleaned_articles = cleaned_articles[cleaned_articles["category"] != "Code"]

### Save final_output df to excel (for troubleshooting)

In [ ]:
import pandas as pd

# Save final_output to csv (Need to save to csv because excel doesn't like datetimes with timezones)
#cleaned_articles.to_csv("cleaned_articles_output.csv", index=False)

# Read in final_output csv
cleaned_articles = pd.read_csv("cleaned_articles_output.csv", low_memory=False)

## SQLite Handling

### Inital creation articles DB in SQLite

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///articles.db")

with engine.begin() as conn:

        # habits db creation
        conn.execute(text("""CREATE TABLE IF NOT EXISTS news_articles (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            title TEXT NOT NULL,
                            link TEXT NOT NULL,
                            source TEXT,
                            language TEXT,
                            published_date TEXT NOT NULL,
                            keywords TEXT,
                            summary TEXT,
                            category TEXT NOT NULL,
                            upload_date TEXT NOT NULL);"""))

### Inserting newly scraped articles into sqlite db

In [11]:
# Reading in article table from sql db
from sqlalchemy import create_engine, text
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd

# Reading in all articles from the db (Could do the date filtering here instead of in python)
engine = create_engine("sqlite:///articles.db")
with engine.connect() as connection:
    articles_db = pd.read_sql_query(
        text("SELECT * FROM news_articles"),connection,
        parse_dates=['published_date', 'upload_date'])
    
# Localizing the dates from sqlite into UTC and converting to 
articles_db["published_date"] = (
    pd.to_datetime(articles_db["published_date"])
    .dt.tz_localize("UTC"))

articles_db["upload_date"] = (
    pd.to_datetime(articles_db["upload_date"])
    .dt.tz_localize("UTC"))

articles_db = articles_db.drop(columns=["id"])

# Using the link variable to check for duplicates and drop them
combined = pd.concat([articles_db, cleaned_articles]).drop_duplicates(subset="link")

# This is the workaround to today not working
final_output = combined

# Below only works when today variable carries over which doesn't work when I read the data from CSV
# Filtering the combined data to only include this batches articles that are not duplicates
#final_output = combined[combined['upload_date'] == today]

In [ ]:
final_output.to_sql(name="news_articles", con=engine, if_exists='append', index=False)

### Article database support table modification

#### Inserting new article sources into article_sources SQLite DB

In [ ]:
# Add 1 link at a time
from sqlalchemy import text, create_engine

engine = create_engine("sqlite:///articles.db")

source = "XML"
link = "https://googlecrawl.npr.org/news/sitemap_news.xml"

with engine.begin() as conn:
    # Insert new article
    conn.execute(
        text("""
            INSERT INTO article_sources 
            (source_type, article_link)
            VALUES (:source_type, :article_link)"""),
        {"source_type": source, "article_link": link})

In [ ]:
# Add a list of links at a time
from sqlalchemy import text, create_engine

engine = create_engine("sqlite:///articles.db")

source = "RSS" #XML RSS

xml_articles = ["https://apnews.com/news-sitemap-content.xml", "https://googlecrawl.npr.org/news/sitemap_news.xml"]
rss_urls =  ["https://github.com/pandas-dev/pandas/releases.atom", "https://github.com/numpy/numpy/releases.atom", "https://github.com/psf/requests/releases.atom", "https://blog.python.org/rss.xml" ]

with engine.begin() as conn:
    for link in rss_urls:
        # Insert new article
        conn.execute(
            text("""
                INSERT INTO article_sources 
                (source_type, article_links, status)
                VALUES (:source_type, :article_links, :status)"""),
            {"source_type": source, "article_links": link, "status": 1})

#### Inserting new email recipients into email_recipients SQL DB

In [ ]:
# Add 1 email at a time
from sqlalchemy import text, create_engine

engine = create_engine("sqlite:///articles.db")

email = "" # define new email here

with engine.begin() as conn:
    # Insert new article
    conn.execute(
        text("""
            INSERT INTO email_recipients 
            (emails)
            VALUES (:emails)"""),
        {"emails": email})

## Converting the pandas df into a markdown file for export

In [12]:
import pandas as pd
from datetime import datetime
from pathlib import Path
import random

final_output2 = final_output

df = final_output2

#def df_to_markdown(df: pd.DataFrame):

# Grabbing unique categories from sqlite db
categories = df['category'].unique().tolist() 

# Build markdown string
markdown_output = f"# 📰 Demolition Daily\n\n"

# list of emojis to itterate through for fun!
news_emojis = ["🆕", "📰", "🌎", "📢", "🖥️", "🗃️", "🗞️"] 

for category in categories:
    subset = df[df["category"] == category]
    if subset.empty:
        continue

    markdown_output += f"""<h2 style="text-decoration: underline; font-size: 28px; margin-top: 30px;">{category}</h2> \n\n"""

    for i, row in subset.iterrows():
        # Handle summary / keywords fallback
        summary_text = None
        if category == "Code":
            summary_text = 'No summary available.'
        else:
        
            if pd.notna(row.get('summary')) and str(row['summary']).strip():
                summary_text = row['summary']
            elif pd.notna(row.get('keywords')) and str(row['keywords']).strip():
                summary_text = row['keywords']
            else:
                summary_text = 'No summary available.'

        # Handle source field gracefully
        source = row.get('source', 'Unknown Source')

        # Build the visually grouped “Card” markdown block
        markdown_output += (
            f"> ### {random.choice(news_emojis)} [{row['title']}]({row['link']})\n"
            f"> **Source:** {source}  \n"
            f"> **Published:** {row['published_date']}  \n"
            f"> **Tags:** {summary_text}\n\n"
        )

# Write to markdown file
output_path = Path("news_digest.md")
output_path.write_text(markdown_output, encoding="utf-8")

print(f"✅ Markdown file generated at: {output_path.resolve()}")


# After uploading final_output to sqlite db I want to convert dates to EST for displaying the news letter
final_output["published_date"] = final_output["published_date"].dt.tz_convert("US/Eastern")  
# Converting published_date to a specific string format for better readability
final_output["published_date"] = final_output["published_date"].dt.strftime("%B %d, %Y %I:%M%p")

# Instead of saving the markdown output in memory, I save it as a file to disk and then read that when sending the emails
#df_to_markdown(final_output)

✅ Markdown file generated at: C:\Users\ryani\Coding Projects\News Aggregator\news_digest.md


## Converting markdown file to HTML and sending out email

In [16]:
import pandas as pd
from sqlalchemy import text

# Grabbing email recipients dynamically from SQL
with engine.connect() as connection:
    email_recipients = pd.read_sql_query(
        text("""SELECT emails 
             FROM email_recipients
             WHERE id = 1""" ), connection)

In [15]:
# Version 1 converting markdown to html and sending that over email  
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import markdown
import pandas as pd
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///articles.db")

# Load variables from .env file
load_dotenv()

email_pass = os.getenv("email_pass")
email_sender = os.getenv("email_sender")

# Grabbing email recipients dynamically from SQL
with engine.connect() as connection:
    email_recipients = pd.read_sql_query(
        text("""SELECT emails FROM email_recipients""" ), connection)
    
email_recipients = email_recipients['emails'].tolist()

# Grabbing today's date as a string for the subject
today_date = datetime.now().strftime("%A, %B %d, %Y")

def send_markdown_as_email(markdown_path: str, email_sender: str, email_pass: str, email_to:list, today_date):
    with open(markdown_path, "r", encoding="utf-8") as f:
        md_text = f.read()

    # Converting markdown file to HTML (actually works pretty well!)
    html = markdown.markdown(md_text)

    msg = MIMEMultipart("alternative")
    msg["From"] = email_sender
    msg["To"] = ", ".join(email_to)  # Convert list to header string
    msg["Subject"] = f"Demolitoin Daily — {today_date}"

    msg.attach(MIMEText(md_text, "plain"))
    msg.attach(MIMEText(html, "html"))

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(email_sender, email_pass)
        server.send_message(msg)

    return print("Email successfully sent!")

send_markdown_as_email("news_digest.md", email_sender, email_pass, email_recipients, today_date)

Email successfully sent!


## Analysis

In [ ]:
# Reading in articles from sql db
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine("sqlite:///articles.db")

with engine.connect() as connection:
    articles = pd.read_sql_query(
        text("""SELECT * 
                from news_articles na 
                --where na.published_date > '2025/12/31'"""), 
        connection)
        #parse_dates=['published_date', 'upload_date']) # Don't need this because I use pd.todatetime below

articles["upload_date"] = (
    pd.to_datetime(articles["upload_date"])
    .dt.tz_localize("UTC"))

articles["published_date"] = (
    pd.to_datetime(articles["published_date"])
    .dt.tz_localize("UTC"))

### Fix (coalescing timezone data)

Delete last modifed date from news_artciles db

In [ ]:
# Reading in articles from sql db
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine("sqlite:///articles.db")

with engine.connect() as connection:
    articles_db = pd.read_sql_query(
        text("""SELECT * 
                from news_articles"""), 
        connection)

In [ ]:
# Assuming all date variables in sqlite db are currently in EST 
articles_db["upload_date"] = (
    pd.to_datetime(articles_db["upload_date"])
    .dt.tz_localize("US/Eastern")
    .dt.tz_convert("UTC"))

articles_db["published_date"] = (
    pd.to_datetime(articles_db["published_date"])
    .dt.tz_localize("US/Eastern")
    .dt.tz_convert("UTC"))

# Filtering out artciles where there is no published date
articles_db = articles_db[articles_db["published_date"].notna()]

articles_db.to_sql(name="news_articles", con=engine, if_exists='replace', index=False)

In [ ]:
# Need to now recreate the habit entries table with correct keys, nulls, and autoincrements
with engine.connect() as conn:
    # habit_entries table
        conn.execute(text("""CREATE TABLE IF NOT EXISTS news_articles_new (
                            id INTEGER PRIMARY KEY AUTOINCREMENT,
                            title TEXT NOT NULL,
                            link TEXT NOT NULL,
                            source TEXT,
                            language TEXT,
                            published_date TEXT NOT NULL,
                            keywords TEXT,
                            summary TEXT,
                            category TEXT NOT NULL,
                            upload_date TEXT NOT NULL);"""))

In [ ]:
with engine.begin() as conn:
    # news_articles table
    conn.execute(text("""INSERT INTO news_articles_new (id, title, link, source, language, published_date, keywords, summary, category, upload_date)
                        SELECT id, title, link, source, language, published_date, keywords, summary, category, upload_date
                        FROM news_articles"""))

# Need to ensure I delete original news_articles last_modified date variable 
# Delete original news_articles table from sqlite and rename "news_articles_new" to "news_articles"

## Extra Code

In [ ]:
import requests
import json

url = "https://query.wikidata.org/sparql"
query = """
        SELECT ?person ?personLabel WHERE {
        ?person wdt:P31 wd:Q5 .          # instance of human
        ?person wdt:P106 ?occupation .   # occupation
        
        VALUES ?occupation {
            wd:Q901   # scientist
            wd:Q82955 # politician
        }
        
        SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
        }
        ORDER BY RAND()
        LIMIT 1
        """

res = requests.get(url, params={"format": "json", "query": query})
data = res.json()

person = data["results"]["bindings"][0]["personLabel"]["value"]
print(person)


### More SQL way of inserting new articles to db

In [ ]:
from sqlalchemy import create_engine, text
from datetime import datetime
import pytz

engine = create_engine("sqlite:///articles.db")

eastern = pytz.timezone('America/New_York')
local_time = datetime.now(eastern).strftime('%Y-%m-%d %H:%M:%S')

with engine.begin() as conn:
    for _, row in final_output.iterrows():
        # Check if article already exists by link
        exists_df = pd.read_sql_query(
            text("SELECT 1 FROM news_articles WHERE link = :link"),
            conn,
            params={"link": row['link']})

        # Properly check if result is empty
        if not exists_df.empty:
            continue

        # Insert new article
        conn.execute(
            text("""
                INSERT INTO news_articles 
                (title, link, source, language, published_date, last_modified, keywords, summary, category, upload_date)
                VALUES (:title, :link, :source, :language, :published_date, :last_modified, :keywords, :summary, :category, :upload_date)
            """),
            {
                "title": row["title"],
                "link": row["link"],
                "source": row["source"],
                "language": row["language"],
                "published_date": row["published_date"],
                "last_modified": row["last_modified"],
                "keywords": row["keywords"],
                "summary": row["summary"],
                "category": row["category"],
                "upload_date": local_time
            },
        )

        print(f"✅ Inserted new article: {row['title']}")

### Langchain LLM try

In [ ]:
import pandas as pd
from langchain_ollama import ChatOllama
from langchain.messages import SystemMessage, HumanMessage 

final_output = pd.read_excel("final_news_output.xlsx")

# Loading in example markdown file as a style guide
with open("template.md", "r") as f:
    markdown_example = f.read()

# Convert DataFrame to a list of dicts, better reasoning for llm & loopings
articles = final_output.to_dict(orient="records")

# Initialize your local LLM
llm = ChatOllama(model="qwen3:latest") 

# Building out system prompt
system_prompt = f"""
You are a professional technical editor who writes weekly news digests in Markdown format.

You'll be given a markdown template to mimic that will news stories bucketed under 5 categories. Each news story will belong to 
one of those cateogies and contains 3 outputs: the artcile title, artcile link, and article summary. You will be given an article in a dictionary 
format such as:

{'title': 'US airlines cancel more than 2,500 weekend flights amid shutdown',
  'link': 'https://apnews.com/article/flight-cancellations-government-shutdown-what-to-know-dea56ba97690b43397a96289f5f949e8',
  'source': 'Associated Press',
  'language': 'eng',
  'published_date': '2025-11-08T00:02:54-05:00',
  'last_modified': '2025-11-08T23:19:54-05:00',
  'keywords': None,
  'category': 'Politics',
  'summary': nan}

yYour job is to position the article in the correct markdown format in the correct news category.

Here is the example markdown file to mimic:

{markdown_example}
"""

# Building out human prompt
human_prompt = f"""
Here are the articles you need to format:

{articles}

Generate a cohesive Markdown file with headings, subheadings, and lists of articles.
Only return valid Markdown.
"""

# ✅ Get the model output
response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=human_prompt)
])

# ✅ Save the final markdown output
with open("weekly_digest.md", "w", encoding="utf-8") as f:
    f.write(response.content)

print("✅ Markdown file generated successfully: weekly_digest.md")


### Traditional web scrapping

In [ ]:
import requests
from bs4 import BeautifulSoup

url = 'https://apnews.com/hub/china'

html = requests.get(url)

soup = BeautifulSoup(html.content, 'html.parser')

In [ ]:
article_list = soup.find_all("div", class_='PageList-items-item') 

In [ ]:
for artcile in article_list:
    title = artcile.find("h1", class_="Headline")
    link = artcile.find("a")['href']
    print(f"Title: {title}, Link: {link}")

In [ ]:
lins

In [ ]:
article_list